In [ ]:
%pip install --upgrade pip
%pip install ipython-sql psycopg2 pandas matplotlib numpy matplotlib

In [ ]:
%load_ext sql

In [ ]:
from urllib.parse import quote_plus
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DB_URL = "postgresql://intern_read_only:%s@api.algotrade.vn:5432/algotradeDB" % quote_plus("Bingo@0711")
%sql $DB_URL

In [ ]:
def query_command(tickersymbol, start_date):
    return """
    select
        a.datetime,
        a.tickersymbol,
        a.price as sell_price,
        b.price as buy_price,
        m.price as matched_price
        from quote.askprice a
    join quote.bidprice b on a.datetime = b.datetime and a.tickersymbol = b.tickersymbol
    join quote."matched" m on a.datetime = m.datetime and a.tickersymbol = m.tickersymbol
    where a.tickersymbol = '{tickersymbol}'
    and a.depth = 1
    and b.depth = 1
    and a.datetime > '{start_date} 00:00:00'
    order by a.datetime;
    """.format(tickersymbol=tickersymbol, start_date=start_date)

# In-sample Data

In [ ]:
in_sample_df = pd.DataFrame()

sql_query = query_command('VN30F2311', '2023-10-24')
result = %sql $sql_query
in_sample_df = pd.concat([in_sample_df, result.DataFrame()], ignore_index=True)

in_sample_df

In [ ]:
# Convert data type
in_sample_df['datetime'] = pd.to_datetime(in_sample_df['datetime'])
in_sample_df['sell_price'] = in_sample_df['sell_price'].astype(float)
in_sample_df['buy_price'] = in_sample_df['buy_price'].astype(float)
in_sample_df['matched_price'] = in_sample_df['matched_price'].astype(float)

In [ ]:
# Plot matched price
in_sample_df.plot(x='datetime', y='matched_price', kind='line', figsize=(16, 8), title="In-sample data - Matched price", xlabel="Date", legend=False)
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# Plot ask price
in_sample_df.plot(x='datetime', y='sell_price', kind='line', figsize=(16, 8), title="In-sample data - Sell price", xlabel="Date", legend=False)
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# Plot bid price
in_sample_df.plot(x='datetime', y='buy_price', kind='line', figsize=(16, 8), title="In-sample data - Buy price", xlabel="Date", legend=False)
plt.gca().spines[['top', 'right']].set_visible(False)

# Out-sample Data

In [ ]:
out_sample_df = pd.DataFrame()

sql_query = query_command('VN30F2312', '2023-11-17')
result = %sql $sql_query
out_sample_df = pd.concat([out_sample_df, result.DataFrame()], ignore_index=True)

out_sample_df

In [ ]:
# Convert data type
out_sample_df['datetime'] = pd.to_datetime(out_sample_df['datetime'])
out_sample_df['sell_price'] = out_sample_df['sell_price'].astype(float)
out_sample_df['buy_price'] = out_sample_df['buy_price'].astype(float)
out_sample_df['matched_price'] = out_sample_df['matched_price'].astype(float)

In [ ]:
# Plot matched price
out_sample_df.plot(x='datetime', y='matched_price', kind='line', figsize=(16, 8), title="Out-sample data - Matched price", xlabel="Date", legend=False)
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# Plot ask price
out_sample_df.plot(x='datetime', y='sell_price', kind='line', figsize=(16, 8), title="Out-sample data - Sell price", xlabel="Date", legend=False)
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# Plot bid price
out_sample_df.plot(x='datetime', y='buy_price', kind='line', figsize=(16, 8), title="Out-sample data - Buy price", xlabel="Date", legend=False)
plt.gca().spines[['top', 'right']].set_visible(False)

# Bollinger bands

In [ ]:
# set number of days and standard deviations to use for Bollinger Bands
window = 10
no_of_std = 1.9

## In-sample

In [ ]:
in_sample_rolling_mean = in_sample_df['matched_price'].rolling(window).mean()
in_sample_rolling_std = in_sample_df['matched_price'].rolling(window).std()

# Calculate three Bollinger Bands
in_sample_df['Middle Band'] = in_sample_rolling_mean
in_sample_df['Upper Band'] = in_sample_rolling_mean + (in_sample_rolling_std * no_of_std)
in_sample_df['Lower Band'] = in_sample_rolling_mean - (in_sample_rolling_std * no_of_std)

In [ ]:
# Plot the Bollinger Bands
in_sample_df[['datetime', 'Middle Band', 'Upper Band', 'Lower Band']].plot(x='datetime', y=['Middle Band', 'Upper Band', 'Lower Band'], kind='line', figsize=(16, 8), title="In-sample Bollinger Bands", xlabel="Date", legend=True)
plt.gca().spines[['top', 'right']].set_visible(False)

## Out-sample

In [ ]:
out_sample_rolling_mean = out_sample_df['matched_price'].rolling(window).mean()
out_sample_rolling_std = out_sample_df['matched_price'].rolling(window).std()

# Calculate three Bollinger Bands
out_sample_df['Middle Band'] = out_sample_rolling_mean
out_sample_df['Upper Band'] = out_sample_rolling_mean + (out_sample_rolling_std * no_of_std)
out_sample_df['Lower Band'] = out_sample_rolling_mean - (out_sample_rolling_std * no_of_std)

In [ ]:
# Plot the Bollinger Bands
out_sample_df[['datetime', 'Middle Band', 'Upper Band', 'Lower Band']].plot(x='datetime', y=['Middle Band', 'Upper Band', 'Lower Band'], kind='line', figsize=(16, 8), title="Out-sample Bollinger Bands", xlabel="Date", legend=True)
plt.gca().spines[['top', 'right']].set_visible(False)